# Phase 253b: Localization geometry control at the 7,513-chain pool: training (geom, geom+topo)

Topology's localization gain is largest on this pool (Phase 86: pooled COACH420 DCC 0.5932 with topology vs 0.4286 without,
8 seeds), and it is the paper's headline localization result. Phases 236-247 (2,055-chain pool) found geometry matches or
beats topology alone at every embedding scale and the residual beyond geometry is not established for localization. This
asks the same question with 3.5x the training data.

Training set = Phase 53's COACH420-clean 7,181 chains (Phase 251, identical for every arm and kernel), true centers from Phase 251,
92/8 train/val split (RandomState 42), Phase 53's model and training loop (lambda 0.1, 40 epochs, patience 8, batch 16, ESM2-35M).
Every arm uses an identical 51-wide side channel (34 topology + 17 geometry columns, unselected zeroed). This kernel trains
arms ['geom', 'geom+topo'] x 8 seeds and saves the checkpoints; the scorer (Phase 254) loads all 32 from the four arms.

## Cell 1: Install

In [ ]:
!pip install -q torch-geometric==2.6.1

## Cell 2: Load the Phase 52 blob, the Phase 251 fixed training list + true centers, and the Phase 250 geometry (3 chunks)

In [ ]:
import gzip, json, os, time, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
blob = torch.load(locate("phase52_graphs.pt"), map_location="cpu", weights_only=False)
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])
by_key = {(str(d.pdb_id), str(d.chain_id)): d for d in blob["dataset"]}
del blob
assert N_PRE == 507 and TOPO_DIM == 34, (N_PRE, TOPO_DIM)

clean = torch.load(locate("phase251_clean_centers.pt"), map_location="cpu", weights_only=False)
feats = {}
for t in "abc":
    g = torch.load(locate(f"phase250{t}_geometry_scale3.pt"), map_location="cpu", weights_only=False)
    feats.update(g["feats"]); print(f"geometry chunk {t}: {g['n_chains']}/{g['n_total']} accepted")
GEOM_DIM = 17; SIDE_DIM = TOPO_DIM + GEOM_DIM

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

dataset, miss_geo = [], 0
for k in clean["order_keys"]:                       # Phase 53's exact chain order
    if k not in feats: miss_geo += 1; continue
    d = by_key[k]; d.true_center = torch.from_numpy(clean["centers"][k]); dataset.append(d)
print(f"clean training chains {len(clean['order_keys']):,}; dropped for missing geometry {miss_geo}; using {len(dataset):,}")
assert miss_geo <= 0.02 * len(clean["order_keys"])

# label-free z-scoring statistics over the training chains
allg = np.concatenate([feats[(str(d.pdb_id), str(d.chain_id))] for d in dataset], 0)
g_mu = torch.from_numpy(allg.mean(0).astype(np.float32)); g_sd = torch.from_numpy(np.maximum(allg.std(0), 1e-6).astype(np.float32))
del allg
for d in dataset:
    f = feats[(str(d.pdb_id), str(d.chain_id))]; assert f.shape == (d.x.size(0), GEOM_DIM)
    d.x = torch.cat([d.x, ((torch.from_numpy(f) - g_mu) / g_sd).clamp(-5, 5)], dim=1)
del feats, by_key
torch.save(dict(g_mu=g_mu, g_sd=g_sd, n_train=len(dataset)), WORK / f"phase253b_geom_stats.pt")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
print(f"node dim {dataset[0].x.size(1)} = {N_PRE} + {TOPO_DIM} + {GEOM_DIM}; train {len(splits['train'])} / val {len(splits['val'])}")

## Cell 3: Model (Phase 53, verbatim)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Training machinery (Phase 53, verbatim except the side-channel width)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, SIDE_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 5: Train arms ['geom', 'geom+topo'] x 8 seeds

In [ ]:
ARMS = ['geom', 'geom+topo']

KEEP_SIDE = {"no-topo": [], "full": list(range(TOPO_DIM)), "geom": list(range(TOPO_DIM, SIDE_DIM)), "geom+topo": list(range(SIDE_DIM))}
SIDE0 = [d.x[:, N_PRE:].clone() for d in dataset]          # original side channels only (the wide ESM block is never copied)
def apply_arm(a):
    m = torch.zeros(SIDE_DIM); m[KEEP_SIDE[a]] = 1.0
    for d, s0 in zip(dataset, SIDE0): d.x[:, N_PRE:] = s0 * m

BEST_LAMBDA = 0.1
train_log = []; t0 = time.time()
for arm in ARMS:
    apply_arm(arm)
    for seed in range(8):
        m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
        v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
        torch.save(m.state_dict(), WORK/f"cache/loc253_{arm.replace('+','_')}_seed{seed}.pt")
        train_log.append({"arm": arm, "seed": seed, "val_pr": float(val_pr), "val_dcc": float(v.get("dcc", float("nan")))})
        json.dump(train_log, open(WORK/"phase253b_train_log.json", "w"), indent=2)
        print(f"  [{arm:9s}] seed {seed}: val PR {val_pr:.4f}  val DCC {v.get('dcc',float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]", flush=True)
print("done; checkpoints in cache/")